In [1]:
# cjpe_issue_analysis_ollama.py
# ─────────────────────────────────────────────────────────────
# QA Generation from ISSUE + ANALYSIS
#
#   • 5 from ISSUE    (legal question framed by court)
#   • 5 from ANALYSIS (court's reasoning and analysis)
#   Total : 10 QA pairs per doc
#
# Provider  : Ollama (local — unlimited, no API key needed)
# Model     : llama3.1 (8B)
# Dataset   : PAVITHRA/legal_judgment_prediction
# Input     : cjpe_track_issue_analysis_cleaned.jsonl
# Output    : issue_analysis_qa_flat_OLLAMA.jsonl
# Checkpoint: issue_analysis_checkpoint_OLLAMA.json  (auto-resume on restart)
#
# Confirmed field names from dataset:
#   ['id', 'ISSUE', 'ANALYSIS', 'label']
#
# LEAKAGE PREVENTION:
#   ANALYSIS text is only read by Ollama LLM to GENERATE questions.
#   Questions are restricted to legal reasoning — not who won.
# ─────────────────────────────────────────────────────────────

import json
import os
import re
import time
import requests
from typing import Optional

# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════
MODEL_ID        = "llama3.1"
OLLAMA_URL      = "http://localhost:11434/api/chat"

INPUT_PATH      = "cjpe_track_issue_analysis_cleaned.jsonl"
OUTPUT_PATH     = "issue_analysis_qa_flat_OLLAMA.jsonl"
CHECKPOINT_PATH = "issue_analysis_checkpoint_OLLAMA.json"

TARGET_DOCS     = 8000

# ── Confirmed field names from dataset ────────────────────────
ISSUE_KEY       = "ISSUE"
ANALYSIS_KEY    = "ANALYSIS"
ID_KEY          = "id"
LABEL_KEY       = "label"

# ══════════════════════════════════════════════════════════════
# PROMPTS
# ══════════════════════════════════════════════════════════════

ISSUE_PROMPT = """You are a legal expert analyzing Indian court judgments.

JUDGMENT OUTCOME: {label_word}

LEGAL ISSUE (key points on which the verdict needs to be delivered, framed by court):
{issue}

Generate exactly 5 question-answer pairs about the legal questions framed by the court.
Each answer MUST end with exactly one of: [FAVORS_PETITIONER] or [FAVORS_RESPONDENT] or [NEUTRAL]

STRICT RULES:
1. Ask about what legal question is being decided, what the court must determine, which right or provision is central.
2. Do NOT ask who won, whether appeal was allowed, or which party succeeded.
3. Issue questions should mostly use [NEUTRAL] — the issue is a question, not an answer.
4. Do NOT use apostrophes or single quotes. Use full words only.
   Example: use "does not" instead of "doesn't", use "courts ruling" instead of "court's ruling".

Good question examples:
- "What is the primary legal question the court must resolve in this case?"
- "Which constitutional right is at the center of the dispute before the court?"
- "What must the court determine regarding the validity of the termination order?"
- "Which legal provision is being challenged in this case?"
- "What is the core dispute the court has been asked to adjudicate?"

Bad question examples (DO NOT ask these):
- "Who won the case?" (reveals verdict)
- "Was the appeal allowed?" (reveals verdict)

Reply with ONLY this JSON, no other text:
{{"Q1":"what legal question must the court decide","A1":"answer [SIGNAL]","Q2":"which right or provision is central to this dispute","A2":"answer [SIGNAL]","Q3":"what must the court determine about the legal issue","A3":"answer [SIGNAL]","Q4":"which legal provision is being challenged","A4":"answer [SIGNAL]","Q5":"what is the core dispute before the court","A5":"answer [SIGNAL]"}}

Replace [SIGNAL] with the correct signal tag for each answer."""


ANALYSIS_PROMPT = """You are a legal expert analyzing Indian court judgments.

JUDGMENT OUTCOME: {label_word}

COURT ANALYSIS (court reasoning, interpretation of law, application of legal principles):
{analysis}

Generate exactly 5 question-answer pairs about the court's legal reasoning and analysis.
Each answer MUST end with exactly one of: [FAVORS_PETITIONER] or [FAVORS_RESPONDENT] or [NEUTRAL]

STRICT RULES:
1. Ask ONLY about: legal principles applied, how the court interpreted the law, what tests or standards the court used, how facts were assessed against the law.
2. Do NOT ask who won, whether appeal was allowed, or which party succeeded.
3. Use [FAVORS_PETITIONER] or [FAVORS_RESPONDENT] only when the reasoning clearly supports one side.
4. Do NOT use apostrophes or single quotes. Use full words only.
   Example: use "does not" instead of "doesn't", use "courts reasoning" instead of "court's reasoning".

Good question examples:
- "What legal standard did the court apply to evaluate the evidence?"
- "How did the court interpret the relevant statutory provision?"
- "Which precedent did the court rely on in its analysis?"
- "What principle did the court use to resolve the conflict between the parties?"
- "How did the court assess the legality of the impugned order?"

Bad question examples (DO NOT ask these):
- "Did the petitioner win?" (reveals verdict)
- "Was the order upheld?" (reveals verdict)

Reply with ONLY this JSON, no other text:
{{"Q1":"what legal standard did the court apply","A1":"answer [SIGNAL]","Q2":"how did the court interpret the relevant provision","A2":"answer [SIGNAL]","Q3":"which precedent or doctrine did the court rely on","A3":"answer [SIGNAL]","Q4":"what principle did the court use to resolve the dispute","A4":"answer [SIGNAL]","Q5":"how did the court assess the legality of the order","A5":"answer [SIGNAL]"}}

Replace [SIGNAL] with the correct signal tag for each answer."""


# ══════════════════════════════════════════════════════════════
# PROMPT BUILDERS
# ══════════════════════════════════════════════════════════════

def label_word(label: int) -> str:
    return "ACCEPTED (Appeal Allowed)" if label == 1 else "REJECTED (Appeal Dismissed)"

def build_issue_prompt(issue: str, label: int) -> str:
    return ISSUE_PROMPT.format(label_word=label_word(label), issue=issue[:1500])

def build_analysis_prompt(analysis: str, label: int) -> str:
    return ANALYSIS_PROMPT.format(label_word=label_word(label), analysis=analysis[:2000])


# ══════════════════════════════════════════════════════════════
# ROBUST JSON EXTRACTOR — 8-layer rescue pipeline
# ══════════════════════════════════════════════════════════════

def clean_raw(raw: str) -> str:
    raw = raw.replace("\\'", " ")
    raw = re.sub(r"```(?:json)?", "", raw)
    return raw.strip()


def extract_json(raw: str) -> dict:
    if not raw:
        return {}

    raw = clean_raw(raw)

    # ── Step 1: extract first { ... } block ───────────────────
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        return regex_fallback(raw)
    raw = match.group(0)

    # ── Step 2: direct parse ──────────────────────────────────
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    # ── Step 3: fix trailing commas ───────────────────────────
    cleaned = re.sub(r",\s*([}\]])", r"\1", raw)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # ── Step 4: collapse newlines inside values ────────────────
    cleaned = re.sub(
        r':\s*"(.*?)"(?=\s*[,}])',
        lambda m: ': "' + re.sub(r'[\n\r]+', ' ', m.group(1)) + '"',
        cleaned,
        flags=re.DOTALL
    )
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # ── Step 5: fix trailing commas again after collapse ──────
    cleaned = re.sub(r",\s*([}\]])", r"\1", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # ── Step 6: replace single quotes with double quotes ──────
    cleaned = cleaned.replace("'", '"')
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # ── Step 7: remove all control characters ─────────────────
    cleaned = re.sub(r'[\x00-\x1f\x7f]', ' ', cleaned)
    cleaned = re.sub(r",\s*([}\]])", r"\1", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # ── Step 8: regex key-value fallback ──────────────────────
    return regex_fallback(raw)


def regex_fallback(raw: str) -> dict:
    """Last resort: extract Q1/A1 keys directly using regex."""
    result = {}

    # Pattern 1: standard  "Q1": "value"
    pattern = r'"(Q\d+|A\d+)"\s*:\s*"(.*?)"(?=\s*[,}\n]|$)'
    matches = re.findall(pattern, raw, re.DOTALL)
    for key, val in matches:
        result[key] = re.sub(r'[\n\r]+', ' ', val).strip()

    if result:
        return result

    # Pattern 2: relaxed — key: value without quotes
    pattern2 = r'(Q\d+|A\d+)\s*:\s*"?(.*?)"?(?=\s*(?:Q\d+|A\d+)\s*:|$)'
    matches2  = re.findall(pattern2, raw, re.DOTALL)
    for key, val in matches2:
        result[key] = re.sub(r'[\n\r]+', ' ', val).strip().strip('"').strip("'")

    return result


# ══════════════════════════════════════════════════════════════
# OLLAMA API CALL — with retry + backoff
# ══════════════════════════════════════════════════════════════

def call_ollama(prompt: str, retries: int = 5) -> dict:
    payload = {
        "model"   : MODEL_ID,
        "messages": [{"role": "user", "content": prompt}],
        "stream"  : False,
        "options" : {
            "temperature"   : 0.1,
            "num_predict"   : 2048,
            "repeat_penalty": 1.1,
        }
    }

    for attempt in range(retries):
        try:
            resp = requests.post(OLLAMA_URL, json=payload, timeout=240)

            if resp.status_code == 404:
                print(f"\n  ❌ Ollama model '{MODEL_ID}' not found.")
                print(f"     Run:  ollama pull {MODEL_ID}")
                raise SystemExit(1)

            if resp.status_code in (502, 503, 504):
                wait = 20 * (attempt + 1)
                print(f"    ⏳ Server error ({resp.status_code}) — waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            data   = resp.json()
            raw    = data["message"]["content"].strip()
            result = extract_json(raw)

            if result:
                return result
            else:
                print(f"    ⚠️  JSON extract failed (attempt {attempt+1}) "
                      f"— raw: {raw[:120]!r}")
                time.sleep(2)

        except requests.exceptions.ConnectionError:
            print(f"\n  ❌ Cannot connect to Ollama. Run:  ollama serve")
            raise SystemExit(1)

        except requests.exceptions.Timeout:
            print(f"    ⚠️  Timeout (attempt {attempt+1}) — retrying...")
            time.sleep(10)

        except SystemExit:
            raise

        except Exception as e:
            print(f"    ⚠️  Error (attempt {attempt+1}): {e}")
            time.sleep(5)

    return {}


# ══════════════════════════════════════════════════════════════
# FLATTEN QA DICT → list of individual QA records
# ══════════════════════════════════════════════════════════════

def flatten_qa(doc_id: str, qa_dict: dict, label: int,
               start_idx: int, source: str) -> list:
    records = []
    n = len([k for k in qa_dict if k.startswith("Q")])

    for i in range(1, n + 1):
        q = qa_dict.get(f"Q{i}", "").strip()
        a = qa_dict.get(f"A{i}", "").strip()

        if not q or not a:
            continue

        signal = "NEUTRAL"
        for tag in ["FAVORS_PETITIONER", "FAVORS_RESPONDENT", "NEUTRAL"]:
            if f"[{tag}]" in a:
                signal = tag
                a = a.replace(f"[{tag}]", "").strip()
                break

        records.append({
            "id"      : f"{doc_id}_Q{start_idx + i}",
            "doc_id"  : doc_id,
            "source"  : source,       # ISSUE / ANALYSIS
            "question": q,
            "answer"  : a,
            "signal"  : signal,
            "label"   : label,
        })

    return records


# ══════════════════════════════════════════════════════════════
# CHECKPOINT HELPERS
# ══════════════════════════════════════════════════════════════

def load_checkpoint() -> Optional[str]:
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH) as f:
            data = json.load(f)
            return data.get("last_doc_id")
    return None

def save_checkpoint(doc_id: str, processed: int):
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump({"last_doc_id": str(doc_id), "processed": processed}, f)


# ══════════════════════════════════════════════════════════════
# OLLAMA SANITY CHECK
# ══════════════════════════════════════════════════════════════

def check_ollama():
    print("  🔍 Checking Ollama connection...")
    try:
        resp    = requests.get("http://localhost:11434/api/tags", timeout=5)
        models  = [m["name"] for m in resp.json().get("models", [])]
        matched = [m for m in models if MODEL_ID in m]
        if not matched:
            print(f"\n  ❌ Model '{MODEL_ID}' not found.")
            print(f"     Available: {models}")
            print(f"     Run: ollama pull {MODEL_ID}")
            raise SystemExit(1)
        print(f"  ✅ Ollama running | Model '{MODEL_ID}' found\n")
    except requests.exceptions.ConnectionError:
        print(f"\n  ❌ Ollama is not running. Start with:  ollama serve")
        raise SystemExit(1)


# ══════════════════════════════════════════════════════════════
# LOAD DATA
# ══════════════════════════════════════════════════════════════

if not os.path.exists(INPUT_PATH):
    print(f"  ❌ Input file not found: {INPUT_PATH}")
    raise SystemExit(1)

records = []
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"\n{'='*60}")
print(f"  ISSUE + ANALYSIS — QA GENERATION")
print(f"{'='*60}")
print(f"  Total docs in file   : {len(records):,}")

# ── CHECKPOINT RESUME ──────────────────────────────────────────
last_doc_id       = load_checkpoint()
already_processed = 0

if last_doc_id is not None:
    ids = [str(r[ID_KEY]) for r in records]
    if last_doc_id in ids:
        resume_idx        = ids.index(last_doc_id) + 1
        already_processed = json.load(open(CHECKPOINT_PATH)).get("processed", resume_idx)
        records           = records[resume_idx:]
        print(f"\n  ♻️  Resuming after '{last_doc_id}'")
        print(f"  ✅ Already done  : {already_processed:,} docs")
        remaining_target  = TARGET_DOCS - already_processed
        if remaining_target <= 0:
            print(f"  ✅ Target already reached!")
            raise SystemExit(0)
        print(f"  📌 Still needed  : {remaining_target:,} docs\n")
    else:
        print(f"  ⚠️  Checkpoint not found in file — fresh start")
        already_processed = 0
else:
    print("  🚀 Fresh start")
    already_processed = 0

# ── STARTUP ────────────────────────────────────────────────────
check_ollama()

sample = records[0] if records else {}
print(f"  📁 Input          : {INPUT_PATH}")
print(f"  💾 Output         : {OUTPUT_PATH}")
print(f"  🤖 Model          : {MODEL_ID} (Ollama local)")
print(f"  🎯 Target docs    : {TARGET_DOCS:,}")
print(f"  📊 QA per doc     : 10  (5 ISSUE + 5 ANALYSIS)")
print(f"  📦 Total QA target: ~{TARGET_DOCS * 10:,} pairs")
print(f"  🔒 Leakage guard  : Questions avoid verdict/outcome\n")
print(f"  Field check on first record:")
print(f"    {ISSUE_KEY:10s}    : {'✅ found' if sample.get(ISSUE_KEY)    else '❌ MISSING'}")
print(f"    {ANALYSIS_KEY:10s} : {'✅ found' if sample.get(ANALYSIS_KEY) else '❌ MISSING'}")
print(f"    {ID_KEY:10s}       : {'✅ found' if sample.get(ID_KEY)       else '❌ MISSING'}")
print(f"    {LABEL_KEY:10s}    : {'✅ found' if LABEL_KEY in sample      else '❌ MISSING'}\n")

if ISSUE_KEY not in sample and ANALYSIS_KEY not in sample:
    print(f"  ❌ Neither '{ISSUE_KEY}' nor '{ANALYSIS_KEY}' found in records.")
    print(f"     Keys found: {list(sample.keys())}")
    raise SystemExit(1)


# ══════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════

out_file   = open(OUTPUT_PATH, "a", encoding="utf-8")
processed  = already_processed
errors     = 0
skipped    = 0
start_time = time.time()

for rec in records:

    # ── EARLY STOP ────────────────────────────────────────────
    if processed >= TARGET_DOCS:
        print(f"\n  🎯 Target of {TARGET_DOCS:,} docs reached — stopping!")
        break

    doc_id   = str(rec[ID_KEY])
    label    = int(rec[LABEL_KEY])
    issue    = rec.get(ISSUE_KEY,    "").strip()
    analysis = rec.get(ANALYSIS_KEY, "").strip()

    # Both fields empty — skip
    if not issue and not analysis:
        skipped += 1
        print(f"  ⚠️  Skipping {doc_id} — both ISSUE and ANALYSIS fields empty")
        continue

    all_flat = []

    # ── ISSUE: 5 questions ────────────────────────────────────
    if issue:
        issue_dict = call_ollama(build_issue_prompt(issue, label))
        if issue_dict:
            all_flat.extend(flatten_qa(doc_id, issue_dict, label,
                                       start_idx=0, source="ISSUE"))
        else:
            print(f"  ⚠️  ISSUE failed for {doc_id}")
    else:
        print(f"  ℹ️  ISSUE empty for {doc_id} — skipping ISSUE block")

    # ── ANALYSIS: 5 questions ─────────────────────────────────
    if analysis:
        analysis_dict = call_ollama(build_analysis_prompt(analysis, label))
        if analysis_dict:
            all_flat.extend(flatten_qa(doc_id, analysis_dict, label,
                                       start_idx=5, source="ANALYSIS"))
        else:
            print(f"  ⚠️  ANALYSIS failed for {doc_id}")
    else:
        print(f"  ℹ️  ANALYSIS empty for {doc_id} — skipping ANALYSIS block")

    # ── Write + Checkpoint ────────────────────────────────────
    if all_flat:
        for flat_rec in all_flat:
            out_file.write(json.dumps(flat_rec, ensure_ascii=False) + "\n")
        out_file.flush()

        processed += 1
        save_checkpoint(doc_id, processed)

        if processed % 100 == 0:
            elapsed = time.time() - start_time
            per_doc = elapsed / max(processed - already_processed, 1)
            eta_hrs = ((TARGET_DOCS - processed) * per_doc) / 3600
            print(f"  ✅ {processed:>5}/{TARGET_DOCS} docs | "
                  f"~{processed*10:,} QA pairs | "
                  f"⏱ {per_doc:.1f}s/doc | "
                  f"ETA: {eta_hrs:.1f} hrs | "
                  f"last: {doc_id}")
    else:
        errors += 1
        print(f"  ⚠️  No QA generated for {doc_id}")

out_file.close()


# ══════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════
elapsed_total = (time.time() - start_time) / 3600

print("\n" + "=" * 60)
print("  ISSUE + ANALYSIS QA GENERATION COMPLETE")
print("=" * 60)
print(f"  Docs processed      : {processed:,} / {TARGET_DOCS:,}")
print(f"  Total QA pairs      : ~{processed * 10:,}")
print(f"  Skipped (both empty): {skipped:,}")
print(f"  Errors              : {errors:,}")
print(f"  Total time          : {elapsed_total:.2f} hrs")
print(f"  Output file         : {OUTPUT_PATH}")
print(f"  Checkpoint file     : {CHECKPOINT_PATH}")

print("\n  Sample output (first 5 QA pairs):")
with open(OUTPUT_PATH) as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        r = json.loads(line)
        print(f"\n  ── Record {i+1} ──────────────────────")
        print(f"  ID     : {r['id']}")
        print(f"  Source : {r['source']}")
        print(f"  Signal : {r['signal']}")
        print(f"  Label  : {r['label']} ({'ACCEPTED' if r['label']==1 else 'REJECTED'})")
        print(f"  Q: {r['question']}")
        print(f"  A: {r['answer']}")


  ISSUE + ANALYSIS — QA GENERATION
  Total docs in file   : 32,229

  ♻️  Resuming after '2011_77'
  ✅ Already done  : 5,015 docs
  📌 Still needed  : 2,985 docs

  🔍 Checking Ollama connection...
  ✅ Ollama running | Model 'llama3.1' found

  📁 Input          : cjpe_track_issue_analysis_cleaned.jsonl
  💾 Output         : issue_analysis_qa_flat_OLLAMA.jsonl
  🤖 Model          : llama3.1 (Ollama local)
  🎯 Target docs    : 8,000
  📊 QA per doc     : 10  (5 ISSUE + 5 ANALYSIS)
  📦 Total QA target: ~80,000 pairs
  🔒 Leakage guard  : Questions avoid verdict/outcome

  Field check on first record:
    ISSUE         : ❌ MISSING
    ANALYSIS   : ❌ MISSING
    id               : ✅ found
    label         : ✅ found

  ⚠️  Skipping 2011_79 — both ISSUE and ANALYSIS fields empty
  ℹ️  ISSUE empty for 2011_81 — skipping ISSUE block
  ℹ️  ISSUE empty for 2011_83 — skipping ISSUE block
  ℹ️  ISSUE empty for 2011_84 — skipping ISSUE block
  ⚠️  Skipping 2011_86 — both ISSUE and ANALYSIS fields empty
